### Load Data

In [1]:
import numpy as np

from helpers import load_csv_data

x_train, x_test, y_train, train_ids, test_ids = load_csv_data(
    "dataset",
    sub_sample=False
)

In [2]:
# Check the shape of the loaded data
print("Training data shape:", x_train.shape)
print("Test data shape:", x_test.shape)
print("Training labels shape:", y_train.shape)

print(train_ids[:5])  # Print the first 5 training IDs
print(test_ids[:5])   # Print the first 5 test IDs
print(test_ids[-5:])  # Print the last 5 test IDs

Training data shape: (328135, 321)
Test data shape: (109379, 321)
Training labels shape: (328135,)
[0 1 2 3 4]
[328135 328136 328137 328138 328139]
[437509 437510 437511 437512 437513]


In [3]:
# Check class imbalance in the training labels
unique, counts = np.unique(y_train, return_counts=True)

print("Label distribution:")
print("-" * 40)

for label, count in zip(unique, counts):
    print(
        f"Label {label}: {count} samples "
        f"({100 * count / len(y_train):.2f}%)"
    )

baseline_accuracy = np.max(counts) / len(y_train)

print(
    f"\nMajority-class baseline accuracy: "
    f"{baseline_accuracy:.4f}"
)

Label distribution:
----------------------------------------
Label -1: 299160 samples (91.17%)
Label 1: 28975 samples (8.83%)

Majority-class baseline accuracy: 0.9117


### Split training data

In [4]:
# We will use the same code we used for ex_3

def split_data(x, y, ratio, seed=1):
    """
    split the dataset based on the split ratio. If ratio is 0.8
    you will have 80% of your data set dedicated to training
    and the rest dedicated to testing. If ratio times the number of samples is not round
    you can use np.floor. Also check the documentation for np.random.permutation,
    it could be useful.

    Args:
        x: numpy array of shape (N,), N is the number of samples.
        y: numpy array of shape (N,).
        ratio: scalar in [0,1]
        seed: integer.

    Returns:
        x_tr: numpy array containing the train data.
        x_te: numpy array containing the test data.
        y_tr: numpy array containing the train labels.
        y_te: numpy array containing the test labels.

    >>> split_data(np.arange(13), np.arange(13), 0.8, 1)
    (array([ 2,  3,  4, 10,  1,  6,  0,  7, 12,  9]), array([ 8, 11,  5]), array([ 2,  3,  4, 10,  1,  6,  0,  7, 12,  9]), array([ 8, 11,  5]))
    """
    # set seed
    np.random.seed(seed)
    # ***************************************************
    # INSERT YOUR CODE HERE
    # split the data based on the given ratio: TODO
    # ***************************************************
    # Randomly shuffle the indices
    indices = np.random.permutation(len(x))

    # Number of training samples
    split_sample = int(np.floor(ratio * len(x)))

    # Split the shuffled indices
    train_indices = indices[:split_sample]
    test_indices = indices[split_sample:]

    # Use SAME indices for x and y
    x_tr = x[train_indices]
    x_te = x[test_indices]
    y_tr = y[train_indices]
    y_te = y[test_indices]

    return x_tr, x_te, y_tr, y_te

In [5]:
train_X, test_X, train_y, test_y = split_data(x_train, y_train, 0.8, seed=2026)

print("Training data shape after split:", train_X.shape)
print("Test data shape after split:", test_X.shape)

Training data shape after split: (262508, 321)
Test data shape after split: (65627, 321)


In [6]:
# Check the train/test split imbalance
def print_label_distribution(y, name):
    values, counts = np.unique(
        y,
        return_counts=True
    )

    print(f"\n{name}")
    print("-" * 40)

    for value, count in zip(values, counts):
        print(
            f"Label {value}: {count} "
            f"({100 * count / len(y):.2f}%)"
        )


print_label_distribution(
    train_y,
    "TRAINING SET"
)

print_label_distribution(
    test_y,
    "VALIDATION SET"
)


TRAINING SET
----------------------------------------
Label -1: 239487 (91.23%)
Label 1: 23021 (8.77%)

VALIDATION SET
----------------------------------------
Label -1: 59673 (90.93%)
Label 1: 5954 (9.07%)


In [7]:
feature_summary = []

num_samples, num_features = train_X.shape

for j in range(num_features):

    col = train_X[:, j]

    missing_mask = np.isnan(col)
    observed = col[~missing_mask]

    n_missing = np.sum(missing_mask)
    n_observed = len(observed)

    missing_fraction = n_missing / num_samples

    if n_observed == 0:
        n_unique = 0
        dominant_value = np.nan
        dominant_fraction = np.nan
        min_value = np.nan
        max_value = np.nan

    else:
        values, counts = np.unique(
            observed,
            return_counts=True
        )

        n_unique = len(values)

        dominant_index = np.argmax(counts)

        dominant_value = values[dominant_index]

        dominant_fraction = (
            counts[dominant_index]
            / n_observed
        )

        min_value = np.min(observed)
        max_value = np.max(observed)

    feature_summary.append({
        "index": j,
        "n_missing": n_missing,
        "n_observed": n_observed,
        "missing_fraction": missing_fraction,
        "n_unique": n_unique,
        "dominant_value": dominant_value,
        "dominant_fraction": dominant_fraction,
        "min": min_value,
        "max": max_value
    })


print(
    f"{'Feature':>8} "
    f"{'Missing%':>10} "
    f"{'Unique':>8} "
    f"{'Dominant':>12} "
    f"{'Dom.%':>10} "
    f"{'Min':>10} "
    f"{'Max':>10}"
)

print("-" * 85)

for s in feature_summary:

    dominant_percentage = (
        100 * s["dominant_fraction"]
        if not np.isnan(s["dominant_fraction"])
        else np.nan
    )

    print(
        f"{s['index']:8d} "
        f"{100*s['missing_fraction']:10.2f} "
        f"{s['n_unique']:8d} "
        f"{s['dominant_value']:12.2f} "
        f"{dominant_percentage:10.2f} "
        f"{s['min']:10.2f} "
        f"{s['max']:10.2f}"
    )

 Feature   Missing%   Unique     Dominant      Dom.%        Min        Max
-------------------------------------------------------------------------------------
       0       0.00       53        20.00       5.30       1.00      72.00
       1       0.00       12         1.00      10.02       1.00      12.00
       2       0.00      414   6022015.00       0.49 1012016.00 12312015.00
       3       0.00       12         3.00       9.58       1.00      12.00
       4       0.00       31         9.00       4.22       1.00      31.00
       5       0.00        2      2015.00      97.52    2015.00    2016.00
       6       0.00        2      1100.00      84.95    1100.00    1200.00
       7       0.00    20750 2015000426.00       0.02 2015000001.00 2015023235.00
       8       0.00    20750 2015000426.00       0.02 2015000001.00 2015023235.00
       9      42.46        1         1.00     100.00       1.00       1.00
      10      42.46        2         1.00      99.98       1.00       2.00

In [8]:
all_missing_features = [
    s["index"]
    for s in feature_summary
    if s["n_observed"] == 0
]

constant_features = [
    s["index"]
    for s in feature_summary
    if s["n_unique"] == 1
]

missing_90 = [
    s["index"]
    for s in feature_summary
    if s["missing_fraction"] >= 0.90
]

missing_95 = [
    s["index"]
    for s in feature_summary
    if s["missing_fraction"] >= 0.95
]

missing_99 = [
    s["index"]
    for s in feature_summary
    if s["missing_fraction"] >= 0.99
]

dominant_99 = [
    s["index"]
    for s in feature_summary
    if (
        not np.isnan(s["dominant_fraction"])
        and s["dominant_fraction"] >= 0.99
    )
]


print("All-missing features:")
print(all_missing_features)
print("Count:", len(all_missing_features))

print("\nConstant when observed:")
print(constant_features)
print("Count:", len(constant_features))

print("\n>= 90% missing:")
print(missing_90)
print("Count:", len(missing_90))

print("\n>= 95% missing:")
print(missing_95)
print("Count:", len(missing_95))

print("\n>= 99% missing:")
print(missing_99)
print("Count:", len(missing_99))

print("\n>= 99% same observed value:")
print(dominant_99)
print("Count:", len(dominant_99))

All-missing features:
[]
Count: 0

Constant when observed:
[9, 11, 12, 18, 19, 22]
Count: 6

>= 90% missing:
[11, 14, 22, 55, 109, 110, 111, 112, 113, 114, 115, 116, 117, 119, 120, 121, 122, 123, 124, 125, 127, 128, 129, 130, 131, 132, 133, 134, 135, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 180, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 194, 195, 196, 197, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215]
Count: 99

>= 95% missing:
[11, 14, 22, 55, 127, 128, 129, 130, 131, 132, 133, 134, 135, 137, 138, 139, 140, 141, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 167, 168, 171, 173, 175, 177, 178, 180, 184, 185, 186, 187, 188, 189, 190, 191, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215]
Count: 68

>= 99% missing:
[11, 14,

In [9]:
print(
    f"{'Feat':>5} "
    f"{'Miss%':>9} "
    f"{'N miss':>9} "
    f"{'N obs':>9} "
    f"{'P(+|miss)':>12} "
    f"{'P(+|obs)':>12} "
    f"{'Diff':>10}"
)

print("-" * 85)

for j in constant_features:

    col = train_X[:, j]

    missing_mask = np.isnan(col)
    observed_mask = ~missing_mask

    n_missing = np.sum(missing_mask)
    n_observed = np.sum(observed_mask)

    if n_missing > 0:
        p_missing = np.mean(
            train_y[missing_mask] == 1
        )
    else:
        p_missing = np.nan

    if n_observed > 0:
        p_observed = np.mean(
            train_y[observed_mask] == 1
        )
    else:
        p_observed = np.nan

    print(
        f"{j:5d} "
        f"{100*n_missing/num_samples:9.2f} "
        f"{n_missing:9d} "
        f"{n_observed:9d} "
        f"{p_missing:12.4f} "
        f"{p_observed:12.4f} "
        f"{p_missing-p_observed:10.4f}"
    )

 Feat     Miss%    N miss     N obs    P(+|miss)     P(+|obs)       Diff
-------------------------------------------------------------------------------------
    9     42.46    111469    151039       0.0619       0.1067    -0.0448
   11     99.99    262485        23       0.0877       0.0435     0.0442
   12     42.46    111470    151038       0.0619       0.1067    -0.0448
   18     57.54    151039    111469       0.1067       0.0619     0.0448
   19     57.54    151039    111469       0.1067       0.0619     0.0448
   22     99.76    261875       633       0.0879       0.0079     0.0800


In [10]:
missing_pattern_groups = {}

for j in range(num_features):

    missing_mask = np.isnan(
        train_X[:, j]
    )

    # Convert boolean mask into compact bytes
    packed_mask = np.packbits(
        missing_mask
    )

    key = packed_mask.tobytes()

    if key not in missing_pattern_groups:
        missing_pattern_groups[key] = []

    missing_pattern_groups[key].append(j)


shared_missing_groups = [
    group
    for group
    in missing_pattern_groups.values()
    if len(group) > 1
]

shared_missing_groups.sort(
    key=len,
    reverse=True
)


print(
    "Number of shared missingness groups:",
    len(shared_missing_groups)
)

print("\nLargest groups:")
print("=" * 60)

for i, group in enumerate(
    shared_missing_groups[:30],
    start=1
):

    print(
        f"Group {i}: "
        f"{len(group)} features"
    )

    print(group)

Number of shared missingness groups: 18

Largest groups:
Group 1: 82 features
[0, 1, 2, 3, 4, 5, 6, 7, 8, 28, 30, 31, 36, 39, 40, 43, 44, 46, 47, 50, 51, 52, 53, 58, 216, 219, 221, 222, 227, 229, 230, 231, 232, 233, 235, 236, 237, 239, 240, 241, 242, 243, 244, 246, 247, 248, 249, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 272, 273, 274, 275, 278, 279, 280, 281, 282, 283, 284, 287, 288, 298, 305, 306, 307, 308, 309, 310, 311, 312, 316, 317]
Group 2: 6 features
[111, 112, 113, 115, 116, 117]
Group 3: 5 features
[9, 10, 13, 54, 56]
Group 4: 4 features
[18, 19, 21, 23]
Group 5: 4 features
[238, 313, 314, 315]
Group 6: 3 features
[37, 38, 234]
Group 7: 3 features
[149, 150, 151]
Group 8: 2 features
[11, 14]
Group 9: 2 features
[32, 33]
Group 10: 2 features
[104, 320]
Group 11: 2 features
[128, 129]
Group 12: 2 features
[133, 134]
Group 13: 2 features
[147, 148]
Group 14: 2 features
[153, 154]
Group 15: 2 features
[186, 187]
Group 16: 2 features
[224, 225]
Group 17: 2 features
[2

In [11]:
MAX_UNIQUE = 10
MIN_CATEGORY_SIZE = 100

low_cardinality_features = [
    s["index"]
    for s in feature_summary
    if (
        s["n_unique"] >= 2
        and s["n_unique"] <= MAX_UNIQUE
    )
]


print(
    "Number of low-cardinality features:",
    len(low_cardinality_features)
)


for j in low_cardinality_features:

    col = train_X[:, j]

    observed = col[~np.isnan(col)]

    values = np.unique(observed)

    print("\n" + "=" * 65)
    print(f"FEATURE {j}")
    print("=" * 65)

    for value in values:

        mask = col == value

        n = np.sum(mask)

        if n < MIN_CATEGORY_SIZE:
            continue

        positive_rate = np.mean(
            train_y[mask] == 1
        )

        print(
            f"value = {value:8.2f} | "
            f"N = {n:7d} | "
            f"P(+1) = {positive_rate:.4f}"
        )

    missing_mask = np.isnan(col)

    n_missing = np.sum(missing_mask)

    if n_missing >= MIN_CATEGORY_SIZE:

        positive_rate_missing = np.mean(
            train_y[missing_mask] == 1
        )

        print(
            f"{'MISSING':>16} | "
            f"N = {n_missing:7d} | "
            f"P(+1) = "
            f"{positive_rate_missing:.4f}"
        )

Number of low-cardinality features: 214

FEATURE 5
value =  2015.00 | N =  256002 | P(+1) = 0.0884
value =  2016.00 | N =    6506 | P(+1) = 0.0613

FEATURE 6
value =  1100.00 | N =  222996 | P(+1) = 0.0898
value =  1200.00 | N =   39512 | P(+1) = 0.0759

FEATURE 10
value =     1.00 | N =  151016 | P(+1) = 0.1067
         MISSING | N =  111469 | P(+1) = 0.0619

FEATURE 13
value =     1.00 | N =   68824 | P(+1) = 0.1024
value =     2.00 | N =   82215 | P(+1) = 0.1103
         MISSING | N =  111469 | P(+1) = 0.0619

FEATURE 14
         MISSING | N =  262485 | P(+1) = 0.0877

FEATURE 20
value =     1.00 | N =   55093 | P(+1) = 0.0774
value =     2.00 | N =   56375 | P(+1) = 0.0468
         MISSING | N =  151040 | P(+1) = 0.1067

FEATURE 21
value =     1.00 | N =  110836 | P(+1) = 0.0622
value =     2.00 | N =     633 | P(+1) = 0.0079
         MISSING | N =  151039 | P(+1) = 0.1067

FEATURE 23
value =     1.00 | N =  104031 | P(+1) = 0.0634
value =     2.00 | N =    7438 | P(+1) = 0.0421
  

In [12]:
candidate_special_codes = np.array([
    7, 8, 9,
    77, 88, 99,
    777, 888, 999,
    7777, 8888, 9999,
    77777, 88888, 99999
], dtype=float)


special_code_features = []

for j in range(num_features):

    col = train_X[:, j]

    observed = col[
        ~np.isnan(col)
    ]

    codes_found = []

    for code in candidate_special_codes:

        if np.any(observed == code):
            codes_found.append(code)

    if len(codes_found) > 0:
        special_code_features.append(
            (j, codes_found)
        )


print(
    "Features containing candidate "
    "special codes:"
)

print("-" * 60)

for feature, codes in special_code_features:

    print(
        f"Feature {feature:3d}: "
        f"{codes}"
    )

Features containing candidate special codes:
------------------------------------------------------------
Feature   0: [np.float64(8.0), np.float64(9.0)]
Feature   1: [np.float64(7.0), np.float64(8.0), np.float64(9.0)]
Feature   3: [np.float64(7.0), np.float64(8.0), np.float64(9.0)]
Feature   4: [np.float64(7.0), np.float64(8.0), np.float64(9.0)]
Feature  15: [np.float64(7.0), np.float64(8.0), np.float64(9.0)]
Feature  16: [np.float64(7.0), np.float64(8.0), np.float64(9.0)]
Feature  17: [np.float64(7.0), np.float64(8.0), np.float64(9.0)]
Feature  24: [np.float64(7.0), np.float64(9.0)]
Feature  25: [np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(77.0), np.float64(99.0)]
Feature  26: [np.float64(7.0), np.float64(9.0)]
Feature  27: [np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(77.0), np.float64(88.0), np.float64(99.0)]
Feature  28: [np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(77.0), np.float64(88.0), np.float64(99.0)]
Feature  29: [np.floa

In [13]:
print("=" * 60)
print("EDA SUMMARY")
print("=" * 60)

print(
    "Total features:",
    num_features
)

print(
    "All-missing features:",
    len(all_missing_features)
)

print(
    "Constant when observed:",
    len(constant_features)
)

print(
    ">= 90% missing:",
    len(missing_90)
)

print(
    ">= 95% missing:",
    len(missing_95)
)

print(
    ">= 99% missing:",
    len(missing_99)
)

print(
    ">= 99% dominated:",
    len(dominant_99)
)

print(
    "Low-cardinality features:",
    len(low_cardinality_features)
)

print(
    "Shared missingness groups:",
    len(shared_missing_groups)
)

print(
    "Features containing candidate "
    "special codes:",
    len(special_code_features)
)

EDA SUMMARY
Total features: 321
All-missing features: 0
Constant when observed: 6
>= 90% missing: 99
>= 95% missing: 68
>= 99% missing: 30
>= 99% dominated: 10
Low-cardinality features: 214
Shared missingness groups: 18
Features containing candidate special codes: 273


In [16]:
# ============================================================
# SELF-CONTAINED MINIMAL BASELINE
# ============================================================
#
# REQUIRED EXISTING VARIABLES:
#
#   train_X   = local training features
#   test_X    = local validation features
#   train_y   = local training labels {-1, +1}
#   test_y    = local validation labels {-1, +1}
#
# This cell:
#
#   1. Uses the raw 321 features
#   2. Drops ONLY columns that cannot sensibly be used:
#        - completely NaN in training
#        - constant after NaN handling
#   3. Median-imputes NaNs
#   4. Standardizes each feature
#   5. DOES NOT add any feature columns
#   6. DOES NOT one-hot encode
#   7. DOES NOT add missing indicators
#   8. DOES NOT manually remove metadata/features
#   9. Trains regularized logistic regression
#  10. Tunes lambda + threshold
#
# ============================================================

import numpy as np
import time

from implementations import (
    reg_logistic_regression,
    sigmoid
)


# ============================================================
# 1. COPY ORIGINAL DATA
# ============================================================

X_tr = train_X.astype(float).copy()
X_val = test_X.astype(float).copy()

y_tr = (train_y == 1).astype(int)
y_val = (test_y == 1).astype(int)


print("=" * 80)
print("RAW DATA")
print("=" * 80)

print("Training shape:  ", X_tr.shape)
print("Validation shape:", X_val.shape)

print(
    "Training positive fraction:  ",
    np.mean(y_tr)
)

print(
    "Validation positive fraction:",
    np.mean(y_val)
)


# ============================================================
# 2. REMOVE ONLY COMPLETELY UNUSABLE COLUMNS
# ============================================================
#
# First remove columns that contain NO observed training value.
#
# We do NOT remove columns based on:
#
#   - missing percentage
#   - feature name
#   - correlation
#   - redundancy
#   - categorical/ordinal status
#
# ============================================================

all_nan = np.all(
    np.isnan(X_tr),
    axis=0
)

keep_mask = ~all_nan

X_tr = X_tr[:, keep_mask]
X_val = X_val[:, keep_mask]


print()
print(
    "Completely-NaN columns removed:",
    np.sum(all_nan)
)

print(
    "Features remaining:",
    X_tr.shape[1]
)


# ============================================================
# 3. MEDIAN IMPUTATION
# ============================================================
#
# IMPORTANT:
#
# Medians come ONLY from training data.
#
# ============================================================

n_features = X_tr.shape[1]

medians = np.zeros(
    n_features,
    dtype=float
)


for j in range(n_features):

    observed = X_tr[
        ~np.isnan(X_tr[:, j]),
        j
    ]

    medians[j] = np.median(
        observed
    )


# Apply TRAINING medians to training and validation.

for j in range(n_features):

    train_missing = np.isnan(
        X_tr[:, j]
    )

    validation_missing = np.isnan(
        X_val[:, j]
    )

    X_tr[
        train_missing,
        j
    ] = medians[j]

    X_val[
        validation_missing,
        j
    ] = medians[j]


print()
print(
    "NaNs after imputation — training:",
    np.isnan(X_tr).sum()
)

print(
    "NaNs after imputation — validation:",
    np.isnan(X_val).sum()
)


# ============================================================
# 4. REMOVE CONSTANT COLUMNS AFTER IMPUTATION
# ============================================================
#
# A column with std = 0 contains no information.
#
# ============================================================

std_before_scaling = np.std(
    X_tr,
    axis=0
)

nonconstant_mask = (
    std_before_scaling > 1e-12
)


num_constant = np.sum(
    ~nonconstant_mask
)


X_tr = X_tr[
    :,
    nonconstant_mask
]

X_val = X_val[
    :,
    nonconstant_mask
]


print()
print(
    "Constant columns removed:",
    num_constant
)

print(
    "Final raw feature count:",
    X_tr.shape[1]
)


# ============================================================
# 5. STANDARDIZATION
# ============================================================
#
# We are NOT creating new features.
#
# We only transform:
#
#           x - mean
#       x = --------
#              std
#
# This is needed so gradient descent behaves properly.
#
# Mean/std come ONLY from training data.
#
# ============================================================

means = np.mean(
    X_tr,
    axis=0
)

stds = np.std(
    X_tr,
    axis=0
)


X_tr = (
    X_tr - means
) / stds


X_val = (
    X_val - means
) / stds


# ============================================================
# 6. FINAL DATA CHECK
# ============================================================

print()
print("=" * 80)
print("MINIMAL PREPROCESSING COMPLETE")
print("=" * 80)

print(
    "Training shape:  ",
    X_tr.shape
)

print(
    "Validation shape:",
    X_val.shape
)

print(
    "Training NaNs:",
    np.isnan(X_tr).sum()
)

print(
    "Validation NaNs:",
    np.isnan(X_val).sum()
)

print(
    "Training infs:",
    np.isinf(X_tr).sum()
)

print(
    "Validation infs:",
    np.isinf(X_val).sum()
)


# ============================================================
# 7. ADD INTERCEPT
# ============================================================

tx_train = np.column_stack([
    np.ones(X_tr.shape[0]),
    X_tr
])

tx_validation = np.column_stack([
    np.ones(X_val.shape[0]),
    X_val
])


print()
print(
    "Training matrix with intercept:",
    tx_train.shape
)

print(
    "Validation matrix with intercept:",
    tx_validation.shape
)


# ============================================================
# 8. METRIC FUNCTION
# ============================================================

def get_metrics(
    y_true,
    probabilities,
    threshold
):

    predictions = (
        probabilities >= threshold
    ).astype(int)


    tp = np.sum(
        (y_true == 1)
        &
        (predictions == 1)
    )

    tn = np.sum(
        (y_true == 0)
        &
        (predictions == 0)
    )

    fp = np.sum(
        (y_true == 0)
        &
        (predictions == 1)
    )

    fn = np.sum(
        (y_true == 1)
        &
        (predictions == 0)
    )


    accuracy = (
        tp + tn
    ) / len(y_true)


    if tp + fp > 0:
        precision = tp / (tp + fp)
    else:
        precision = 0.0


    if tp + fn > 0:
        recall = tp / (tp + fn)
    else:
        recall = 0.0


    if precision + recall > 0:

        f1 = (
            2
            * precision
            * recall
            /
            (precision + recall)
        )

    else:

        f1 = 0.0


    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "positive_fraction":
            np.mean(predictions)
    }


# ============================================================
# 9. THRESHOLDS
# ============================================================

thresholds = np.arange(
    0.05,
    0.401,
    0.005
)


# ============================================================
# 10. HYPERPARAMETERS
# ============================================================

lambda_values = [
    0.0,
    1e-6,
    1e-5,
    1e-4,
    1e-3
]


gamma = 0.3

max_iters = 1000


# ============================================================
# 11. TRAIN MODELS
# ============================================================

results = []


print()
print("=" * 80)
print("TRAINING MINIMAL BASELINE")
print("=" * 80)


for model_index, lambda_ in enumerate(
    lambda_values
):

    print()
    print("-" * 80)

    print(
        f"MODEL {model_index + 1}/"
        f"{len(lambda_values)}"
    )

    print(
        f"lambda = {lambda_}"
    )

    print(
        f"gamma = {gamma}"
    )

    print(
        f"iterations = {max_iters}"
    )


    initial_w = np.zeros(
        tx_train.shape[1]
    )


    start_time = time.time()


    w, loss = reg_logistic_regression(
        y_tr,
        tx_train,
        lambda_,
        initial_w,
        max_iters,
        gamma
    )


    elapsed = (
        time.time()
        -
        start_time
    )


    # ========================================================
    # CHECK WEIGHTS
    # ========================================================

    if not np.all(
        np.isfinite(w)
    ):

        print(
            "Weights became non-finite."
        )

        print(
            "Skipping model."
        )

        continue


    # ========================================================
    # PROBABILITIES
    # ========================================================

    train_prob = sigmoid(
        tx_train @ w
    )

    val_prob = sigmoid(
        tx_validation @ w
    )


    if (
        not np.all(np.isfinite(train_prob))
        or
        not np.all(np.isfinite(val_prob))
    ):

        print(
            "Probabilities became non-finite."
        )

        print(
            "Skipping model."
        )

        continue


    # ========================================================
    # FIND BEST VALIDATION THRESHOLD
    # ========================================================

    best_val_metrics = None
    best_threshold = None


    for threshold in thresholds:

        metrics = get_metrics(
            y_val,
            val_prob,
            threshold
        )


        if (
            best_val_metrics is None
            or
            metrics["f1"]
            >
            best_val_metrics["f1"]
        ):

            best_val_metrics = metrics

            best_threshold = threshold


    # ========================================================
    # TRAIN METRICS AT SAME THRESHOLD
    # ========================================================

    train_metrics = get_metrics(
        y_tr,
        train_prob,
        best_threshold
    )


    # ========================================================
    # SAVE RESULT
    # ========================================================

    results.append({

        "lambda":
            lambda_,

        "threshold":
            best_threshold,

        "loss":
            loss,

        "train_accuracy":
            train_metrics["accuracy"],

        "train_precision":
            train_metrics["precision"],

        "train_recall":
            train_metrics["recall"],

        "train_f1":
            train_metrics["f1"],

        "val_accuracy":
            best_val_metrics["accuracy"],

        "val_precision":
            best_val_metrics["precision"],

        "val_recall":
            best_val_metrics["recall"],

        "val_f1":
            best_val_metrics["f1"],

        "tp":
            best_val_metrics["tp"],

        "tn":
            best_val_metrics["tn"],

        "fp":
            best_val_metrics["fp"],

        "fn":
            best_val_metrics["fn"],

        "positive_fraction":
            best_val_metrics[
                "positive_fraction"
            ],

        "weights":
            w.copy(),

        "time":
            elapsed
    })


    print(
        f"Training time: "
        f"{elapsed:.1f}s"
    )

    print(
        f"Best threshold: "
        f"{best_threshold:.3f}"
    )

    print(
        f"Validation F1: "
        f"{best_val_metrics['f1']:.4f}"
    )

    print(
        f"Validation accuracy: "
        f"{best_val_metrics['accuracy']:.4f}"
    )

    print(
        f"Precision: "
        f"{best_val_metrics['precision']:.4f}"
    )

    print(
        f"Recall: "
        f"{best_val_metrics['recall']:.4f}"
    )


# ============================================================
# 12. CHECK THAT MODELS WORKED
# ============================================================

if len(results) == 0:

    raise RuntimeError(
        "Every model failed. "
        "No valid result was produced."
    )


# ============================================================
# 13. SORT BY VALIDATION F1
# ============================================================

results.sort(
    key=lambda x:
        x["val_f1"],
    reverse=True
)


# ============================================================
# 14. PRINT RANKING
# ============================================================

print()
print("=" * 100)
print("FINAL RESULTS")
print("=" * 100)

print(
    f"{'Rank':>5}"
    f"{'Lambda':>12}"
    f"{'Thresh':>10}"
    f"{'Train F1':>12}"
    f"{'Val F1':>12}"
    f"{'Val Acc':>12}"
    f"{'Precision':>12}"
    f"{'Recall':>12}"
)

print("-" * 100)


for rank, result in enumerate(
    results,
    start=1
):

    print(
        f"{rank:5d}"
        f"{result['lambda']:12.1e}"
        f"{result['threshold']:10.3f}"
        f"{result['train_f1']:12.4f}"
        f"{result['val_f1']:12.4f}"
        f"{result['val_accuracy']:12.4f}"
        f"{result['val_precision']:12.4f}"
        f"{result['val_recall']:12.4f}"
    )


# ============================================================
# 15. BEST MODEL
# ============================================================

best_model = results[0]


print()
print("=" * 80)
print("BEST MINIMAL BASELINE")
print("=" * 80)

print(
    f"Lambda:     "
    f"{best_model['lambda']}"
)

print(
    f"Threshold:  "
    f"{best_model['threshold']:.3f}"
)

print(
    f"Accuracy:   "
    f"{best_model['val_accuracy']:.4f}"
)

print(
    f"Precision:  "
    f"{best_model['val_precision']:.4f}"
)

print(
    f"Recall:     "
    f"{best_model['val_recall']:.4f}"
)

print(
    f"F1:         "
    f"{best_model['val_f1']:.4f}"
)

print(
    f"TP={best_model['tp']}  "
    f"TN={best_model['tn']}  "
    f"FP={best_model['fp']}  "
    f"FN={best_model['fn']}"
)


# ============================================================
# 16. COMPARE WITH PREVIOUS PIPELINE
# ============================================================

previous_f1 = 0.4419
previous_accuracy = 0.8721


print()
print("=" * 80)
print("COMPARISON WITH PREVIOUS PIPELINE")
print("=" * 80)

print(
    f"Previous F1:       "
    f"{previous_f1:.4f}"
)

print(
    f"Minimal F1:        "
    f"{best_model['val_f1']:.4f}"
)

print(
    f"Difference:        "
    f"{best_model['val_f1'] - previous_f1:+.4f}"
)


print()

print(
    f"Previous accuracy: "
    f"{previous_accuracy:.4f}"
)

print(
    f"Minimal accuracy:  "
    f"{best_model['val_accuracy']:.4f}"
)

print(
    f"Difference:        "
    f"{best_model['val_accuracy'] - previous_accuracy:+.4f}"
)

RAW DATA
Training shape:   (262508, 321)
Validation shape: (65627, 321)
Training positive fraction:   0.08769637496762003
Validation positive fraction: 0.09072485409968459

Completely-NaN columns removed: 0
Features remaining: 321

NaNs after imputation — training: 0
NaNs after imputation — validation: 0

Constant columns removed: 6
Final raw feature count: 315

MINIMAL PREPROCESSING COMPLETE
Training shape:   (262508, 315)
Validation shape: (65627, 315)
Training NaNs: 0
Validation NaNs: 0
Training infs: 0
Validation infs: 0

Training matrix with intercept: (262508, 316)
Validation matrix with intercept: (65627, 316)

TRAINING MINIMAL BASELINE

--------------------------------------------------------------------------------
MODEL 1/5
lambda = 0.0
gamma = 0.3
iterations = 1000
Training time: 37.5s
Best threshold: 0.195
Validation F1: 0.4251
Validation accuracy: 0.8660
Precision: 0.3480
Recall: 0.5460

--------------------------------------------------------------------------------
MODEL

### Training

In [15]:
# ============================================================
# MINIMAL-PREPROCESSING LOGISTIC REGRESSION BASELINE
# ============================================================

import numpy as np
import time

from implementations import (
    reg_logistic_regression,
    sigmoid
)


# ============================================================
# 1. ADD INTERCEPT
# ============================================================

tx_train = np.column_stack([
    np.ones(
        train_X_processed.shape[0]
    ),
    train_X_processed
])


tx_validation = np.column_stack([
    np.ones(
        test_X_processed.shape[0]
    ),
    test_X_processed
])


y_tr = train_y_logistic
y_val = test_y_logistic


print(
    "Training matrix:",
    tx_train.shape
)

print(
    "Validation matrix:",
    tx_validation.shape
)


# ============================================================
# 2. METRICS
# ============================================================

def get_classification_metrics(
    y_true,
    probabilities,
    threshold
):

    y_pred = (
        probabilities >= threshold
    ).astype(int)


    tp = np.sum(
        (y_true == 1) &
        (y_pred == 1)
    )

    tn = np.sum(
        (y_true == 0) &
        (y_pred == 0)
    )

    fp = np.sum(
        (y_true == 0) &
        (y_pred == 1)
    )

    fn = np.sum(
        (y_true == 1) &
        (y_pred == 0)
    )


    accuracy = (
        (tp + tn)
        /
        len(y_true)
    )


    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )


    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )


    f1 = (
        2 * precision * recall
        /
        (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )


    return {

        "threshold":
            threshold,

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "tp":
            int(tp),

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "predicted_positive_fraction":
            np.mean(y_pred == 1),
    }


# ============================================================
# 3. THRESHOLD SEARCH
# ============================================================

thresholds = np.arange(
    0.05,
    0.4001,
    0.005
)


def find_best_threshold(
    y_true,
    probabilities
):

    best = None


    for threshold in thresholds:

        result = (
            get_classification_metrics(
                y_true,
                probabilities,
                threshold
            )
        )


        if (
            best is None
            or
            result["f1"] > best["f1"]
        ):

            best = result


    return best


# ============================================================
# 4. HYPERPARAMETERS
# ============================================================
#
# We keep gamma and iterations fixed at the best values from
# the previous experiment.
#
# That lets us primarily compare:
#
#       OLD preprocessing
#
# against
#
#       MINIMAL preprocessing
#
# ============================================================

lambda_values = [
    0.0,
    1e-6,
    1e-5,
    1e-4,
    1e-3,
]


gamma = 0.3
max_iters = 1000


# ============================================================
# 5. TRAIN MODELS
# ============================================================

results = []


print("\n")
print("=" * 90)
print("MINIMAL-PREPROCESSING BASELINE")
print("=" * 90)


for model_number, lambda_ in enumerate(
    lambda_values,
    start=1
):

    print("\n")
    print("=" * 90)

    print(
        f"MODEL "
        f"{model_number}/"
        f"{len(lambda_values)}"
    )

    print("=" * 90)


    print(
        f"lambda     = {lambda_}"
    )

    print(
        f"gamma      = {gamma}"
    )

    print(
        f"iterations = {max_iters}"
    )


    initial_w = np.zeros(
        tx_train.shape[1]
    )


    start = time.time()


    w, loss = reg_logistic_regression(
        y_tr,
        tx_train,
        lambda_,
        initial_w,
        max_iters,
        gamma
    )


    elapsed = (
        time.time() - start
    )


    # ========================================================
    # NUMERICAL CHECK
    # ========================================================

    if (
        np.any(np.isnan(w))
        or
        np.any(np.isinf(w))
        or
        np.isnan(loss)
        or
        np.isinf(loss)
    ):

        print(
            "\nNumerical instability."
        )

        print(
            "Skipping model."
        )

        continue


    # ========================================================
    # PROBABILITIES
    # ========================================================

    train_prob = sigmoid(
        tx_train @ w
    )

    validation_prob = sigmoid(
        tx_validation @ w
    )


    # ========================================================
    # OPTIMIZE THRESHOLD ON VALIDATION
    # ========================================================

    validation_result = (
        find_best_threshold(
            y_val,
            validation_prob
        )
    )


    best_threshold = (
        validation_result[
            "threshold"
        ]
    )


    # ========================================================
    # TRAIN PERFORMANCE AT SAME THRESHOLD
    # ========================================================

    training_result = (
        get_classification_metrics(
            y_tr,
            train_prob,
            best_threshold
        )
    )


    # ========================================================
    # SAVE
    # ========================================================

    result = {

        "lambda":
            lambda_,

        "gamma":
            gamma,

        "iterations":
            max_iters,

        "threshold":
            best_threshold,

        "loss":
            loss,

        "train_accuracy":
            training_result[
                "accuracy"
            ],

        "train_precision":
            training_result[
                "precision"
            ],

        "train_recall":
            training_result[
                "recall"
            ],

        "train_f1":
            training_result[
                "f1"
            ],

        "validation_accuracy":
            validation_result[
                "accuracy"
            ],

        "validation_precision":
            validation_result[
                "precision"
            ],

        "validation_recall":
            validation_result[
                "recall"
            ],

        "validation_f1":
            validation_result[
                "f1"
            ],

        "tp":
            validation_result["tp"],

        "tn":
            validation_result["tn"],

        "fp":
            validation_result["fp"],

        "fn":
            validation_result["fn"],

        "predicted_positive_fraction":
            validation_result[
                "predicted_positive_fraction"
            ],

        "time":
            elapsed,

        "weights":
            w.copy(),
    }


    results.append(result)


    # ========================================================
    # PRINT RESULT
    # ========================================================

    print(
        f"\nLoss: "
        f"{loss:.6f}"
    )

    print(
        f"Training time: "
        f"{elapsed:.1f} seconds"
    )

    print(
        f"\nBest threshold: "
        f"{best_threshold:.3f}"
    )


    print("\nTRAINING")

    print(
        f"Accuracy:  "
        f"{training_result['accuracy']:.4f}"
    )

    print(
        f"Precision: "
        f"{training_result['precision']:.4f}"
    )

    print(
        f"Recall:    "
        f"{training_result['recall']:.4f}"
    )

    print(
        f"F1:        "
        f"{training_result['f1']:.4f}"
    )


    print("\nVALIDATION")

    print(
        f"Accuracy:  "
        f"{validation_result['accuracy']:.4f}"
    )

    print(
        f"Precision: "
        f"{validation_result['precision']:.4f}"
    )

    print(
        f"Recall:    "
        f"{validation_result['recall']:.4f}"
    )

    print(
        f"F1:        "
        f"{validation_result['f1']:.4f}"
    )


# ============================================================
# 6. RANK RESULTS
# ============================================================

results = sorted(
    results,
    key=lambda x:
        x["validation_f1"],
    reverse=True
)


print("\n")
print("=" * 110)
print("FINAL RANKING")
print("=" * 110)


print(
    f"{'Rank':>5}"
    f"{'Lambda':>12}"
    f"{'Thresh':>12}"
    f"{'Train F1':>12}"
    f"{'Val F1':>12}"
    f"{'Val Acc':>12}"
    f"{'Precision':>12}"
    f"{'Recall':>12}"
)


print("-" * 110)


for rank, result in enumerate(
    results,
    start=1
):

    print(
        f"{rank:5d}"
        f"{result['lambda']:12.1e}"
        f"{result['threshold']:12.3f}"
        f"{result['train_f1']:12.4f}"
        f"{result['validation_f1']:12.4f}"
        f"{result['validation_accuracy']:12.4f}"
        f"{result['validation_precision']:12.4f}"
        f"{result['validation_recall']:12.4f}"
    )


# ============================================================
# 7. BEST MINIMAL MODEL
# ============================================================

best_minimal_model = results[0]


print("\n")
print("=" * 90)
print("BEST MINIMAL-PREPROCESSING MODEL")
print("=" * 90)


print(
    f"\nLambda:    "
    f"{best_minimal_model['lambda']}"
)

print(
    f"Threshold: "
    f"{best_minimal_model['threshold']:.3f}"
)


print("\nVALIDATION")

print(
    f"Accuracy:  "
    f"{best_minimal_model['validation_accuracy']:.4f}"
)

print(
    f"Precision: "
    f"{best_minimal_model['validation_precision']:.4f}"
)

print(
    f"Recall:    "
    f"{best_minimal_model['validation_recall']:.4f}"
)

print(
    f"F1:        "
    f"{best_minimal_model['validation_f1']:.4f}"
)


print(
    f"\nTP={best_minimal_model['tp']}, "
    f"TN={best_minimal_model['tn']}, "
    f"FP={best_minimal_model['fp']}, "
    f"FN={best_minimal_model['fn']}"
)


# ============================================================
# 8. DIRECT COMPARISON AGAINST OUR CURRENT BEST
# ============================================================

previous_best_f1 = 0.4419
previous_best_accuracy = 0.8721


print("\n")
print("=" * 90)
print("MINIMAL PREPROCESSING VS PREVIOUS PIPELINE")
print("=" * 90)


print(
    f"\nPrevious F1:        "
    f"{previous_best_f1:.4f}"
)

print(
    f"Minimal F1:         "
    f"{best_minimal_model['validation_f1']:.4f}"
)

print(
    f"F1 difference:      "
    f"{best_minimal_model['validation_f1'] - previous_best_f1:+.4f}"
)


print(
    f"\nPrevious accuracy:  "
    f"{previous_best_accuracy:.4f}"
)

print(
    f"Minimal accuracy:   "
    f"{best_minimal_model['validation_accuracy']:.4f}"
)

print(
    f"Accuracy difference:"
    f"{best_minimal_model['validation_accuracy'] - previous_best_accuracy:+.4f}"
)

Training matrix: (262508, 305)
Validation matrix: (65627, 305)


MINIMAL-PREPROCESSING BASELINE


MODEL 1/5
lambda     = 0.0
gamma      = 0.3
iterations = 1000


c:\Users\matsa\Downloads\WS-2026-ML\EX\ML_course\projects\project1\implementations.py:72: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))
c:\Users\matsa\Downloads\WS-2026-ML\EX\ML_course\projects\project1\implementations.py:67: RuntimeWarning: divide by zero encountered in log
  loss = -np.mean(y*np.log(sigmoid(tx@initial_w)) + (1-y)*np.log(1-sigmoid(tx@initial_w)))
c:\Users\matsa\Downloads\WS-2026-ML\EX\ML_course\projects\project1\implementations.py:67: RuntimeWarning: invalid value encountered in multiply
  loss = -np.mean(y*np.log(sigmoid(tx@initial_w)) + (1-y)*np.log(1-sigmoid(tx@initial_w)))



Numerical instability.
Skipping model.


MODEL 2/5
lambda     = 1e-06
gamma      = 0.3
iterations = 1000

Numerical instability.
Skipping model.


MODEL 3/5
lambda     = 1e-05
gamma      = 0.3
iterations = 1000

Numerical instability.
Skipping model.


MODEL 4/5
lambda     = 0.0001
gamma      = 0.3
iterations = 1000

Numerical instability.
Skipping model.


MODEL 5/5
lambda     = 0.001
gamma      = 0.3
iterations = 1000

Numerical instability.
Skipping model.


FINAL RANKING
 Rank      Lambda      Thresh    Train F1      Val F1     Val Acc   Precision      Recall
--------------------------------------------------------------------------------------------------------------


IndexError: list index out of range